# Transformation 01 — Type Casting & Ordinal Encoding

1. **Risk → ordinal**: `Low` -> 1, `Medium` -> 2, `High` -> 3
2. **Date columns → datetime**: Convert `LICENSE TERM START DATE`, `LICENSE TERM EXPIRATION DATE`,
   and `Inspection Date` to datetime; extract temporal features.
3. **License duration**: Compute `license_duration_days` = expiration − start.
4. **Handle missing dates**: FLAG missing `LICENSE TERM START DATE` values;
   impute with median where needed.

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.transformed('train_t00.parquet'))
val   = pd.read_parquet(DataLoader.transformed('val_t00.parquet'))
test  = pd.read_parquet(DataLoader.transformed('test_t00.parquet'))
print('Train shape:', train.shape)
print('Val   shape:', val.shape)
print('Test  shape:', test.shape)
print()
print('Train columns:', list(train.columns))

## 1 · Risk — Ordinal Encoding

Map the `Risk` column to numeric ordinal values:
- `High` -> 3
- `Medium` -> 2
- `Low` -> 1

In [ ]:
print('=== Risk unique values (before) ===')
print(train['Risk'].value_counts())
print()

RISK_MAP = {
    'High': 3,
    'Medium': 2,
    'Low': 1,
    'Unknown': 3  # mode imputation for missing values
}

for split_name, split_df in [('train', train), ('val', val), ('test', test)]:
    unexpected = set(split_df['Risk'].dropna().unique()) - set(RISK_MAP.keys())
    if unexpected:
        print(f'WARNING: unexpected Risk values in {split_name}: {unexpected}')

train['Risk'] = train['Risk'].map(RISK_MAP)
val['Risk']   = val['Risk'].map(RISK_MAP)
test['Risk']  = test['Risk'].map(RISK_MAP)

print('=== Risk unique values (after) ===')
print(train['Risk'].value_counts())

## 2 · Date columns — convert to datetime & extract features

In [ ]:
DATE_COLS = ['Inspection Date', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE']

for col in DATE_COLS:
    if col in train.columns:
        train[col] = pd.to_datetime(train[col], errors='coerce')
        val[col]   = pd.to_datetime(val[col],   errors='coerce')
        test[col]  = pd.to_datetime(test[col],  errors='coerce')
        print(f'{col}: dtype={train[col].dtype}, '
              f'nulls_train={train[col].isna().sum()}, '
              f'nulls_val={val[col].isna().sum()}, '
              f'nulls_test={test[col].isna().sum()}')

### 2a · Inspection Date features

In [ ]:
for df in [train, val, test]:
    df['inspection_year']       = df['Inspection Date'].dt.year
    df['inspection_month']      = df['Inspection Date'].dt.month
    df['inspection_dayofweek']  = df['Inspection Date'].dt.dayofweek
    df['inspection_quarter']    = df['Inspection Date'].dt.quarter

print('Inspection Date features extracted.')
print(train[['inspection_year', 'inspection_month', 'inspection_dayofweek', 'inspection_quarter']].describe())

### 2b · License duration feature

Compute `license_duration_days` = `LICENSE TERM EXPIRATION DATE` − `LICENSE TERM START DATE`.
Also flag rows where `LICENSE TERM START DATE` was missing.

In [ ]:
print('=== LICENSE TERM EXPIRATION DATE nulls ===')
print(f'Train: {train["LICENSE TERM EXPIRATION DATE"].isna().sum()}')
print(f'Val:   {val["LICENSE TERM EXPIRATION DATE"].isna().sum()}')
print(f'Test:  {test["LICENSE TERM EXPIRATION DATE"].isna().sum()}')

print()
print('=== Inspection Date nulls ===')
print(f'Train: {train["Inspection Date"].isna().sum()}')
print(f'Val:   {val["Inspection Date"].isna().sum()}')
print(f'Test:  {test["Inspection Date"].isna().sum()}')

In [ ]:
# days_to_license_expiry: negative = license was expired at inspection time
for df in [train, val, test]:
    df['days_to_license_expiry'] = (
        pd.to_datetime(df['LICENSE TERM EXPIRATION DATE']) - pd.to_datetime(df['Inspection Date'])
    ).dt.days

# Flag missing before filling
for df in [train, val, test]:
    df['license_expiry_missing'] = df['days_to_license_expiry'].isna().astype(int)

# Fit median on train only, apply to val and test
median_expiry = train['days_to_license_expiry'].median()
print(f'Median days to license expiry (train): {median_expiry:.0f} days')

train['days_to_license_expiry'] = train['days_to_license_expiry'].fillna(median_expiry)
val['days_to_license_expiry']   = val['days_to_license_expiry'].fillna(median_expiry)
test['days_to_license_expiry']  = test['days_to_license_expiry'].fillna(median_expiry)

print(f'Remaining nulls — train: {train["days_to_license_expiry"].isna().sum()}, '
      f'val: {val["days_to_license_expiry"].isna().sum()}, '
      f'test: {test["days_to_license_expiry"].isna().sum()}')

## 3 · Save intermediate results

In [ ]:
train.to_parquet(DataLoader.transformed('train_t01.parquet'), index=False)
val.to_parquet(DataLoader.transformed('val_t01.parquet'),   index=False)
test.to_parquet(DataLoader.transformed('test_t01.parquet'),  index=False)
print('Saved train_t01.parquet, val_t01.parquet, and test_t01.parquet')
print('Train shape:', train.shape)
print('Val   shape:', val.shape)
print('Test  shape:', test.shape)